# End-to-End MLOps: ISO New England Energy Demand Forecasting

## Project Overview
This notebook is the first phase in building a complete, production-ready MLOps pipeline for time-series forecasting. Because electricity cannot be easily stored at scale, grid operators must predict hourly power demand precisely to match generation and prevent blackouts.

Our primary goal is not just model accuracy, but **system architecture**. We are evaluating the dataset to build a pipeline that can handle automated ingestion, real-time telemetry, and continuous monitoring.

**Core Objectives for this Notebook:**
1. **Data Feasibility:** Ingest and validate the ISO-NE time-series data.
2. **Feature Engineering:** Implement strict lagging strategies (e.g., 48-hour lag) to prevent data leakage between Day-Ahead (batch) and Real-Time (streaming) inference.
3. **Model Prototyping:** Train a baseline XGBoost model that can eventually be deployed as a unified artifact for both Day-Ahead and Real-Time prediction.

## Dataset Information
* **Domain:** Energy / Autonomous Grid Balancing
* **Target Variable:** System Load (Megawatts)
* **Source:** [ISO New England (ISO Express) - Load & Demand Data](https://www.iso-ne.com/isoexpress/web/reports/load-and-demand)

# 🛠️ 1.  Environment Setup

In [1]:
# imports for production ML
import warnings
warnings.filterwarnings('ignore')

# 🔧 Core Python libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from datetime import datetime
import os
import time

# 🧰 Sklearn Core & Linear Baselines
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, ElasticNet

# 🌲 Ensemble & Boosting Models
from sklearn.ensemble import (
    RandomForestRegressor, 
    GradientBoostingRegressor, 
    AdaBoostRegressor, 
    HistGradientBoostingRegressor
)
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# 📊 Evaluation Metrics
from sklearn.metrics import (
    mean_squared_error, 
    r2_score, 
    mean_absolute_error, 
    mean_absolute_percentage_error
)

#  2. Configuration & Reproducibility

In [ ]:

class Config:
    """
    Centralized configuration for ISO-NE Energy Demand Forecasting.
    All hyperparameters, paths, and constants defined in one place.
    """
    
    # ============ REPRODUCIBILITY ============
    RANDOM_STATE = 42
    TEST_SIZE = 0.2
    VAL_SIZE = 0.2  # 20% of training data (16% of total)
    CV_FOLDS = 5
    N_JOBS = -1
    
    # ============ DIRECTORIES ============
    MODEL_DIR = "models_iso"
    DATA_DIR = "data_iso"
    
    # ============ TARGET & DOMAIN CONSTANTS ============
    TARGET_COL = "DEMAND"          # Target variable (Megawatts)
    MIN_DEMAND_MW = 5000           # Grid demand in New England almost never drops below 10,000 MW, even in the middle of the night on mild spring days. However, because of "Behind-the-Meter" solar panels (people generating their own solar power at home), the grid sometimes "sees" very low demand.
    MAX_DEMAND_MW = 35000          # The all-time record for peak electricity demand in New England's history was 28,130 MW. This occurred on August 2, 2006, during an extreme summer heat wave.  
    # ============ OUTLIER HANDLING ============
    IQR_FACTOR = 1.5
    
    # ============ MODEL HYPERPARAMETERS (BASELINES) ============
    # Linear Baselines
    RIDGE_ALPHA = 1.0
    ELASTIC_ALPHA = 0.1
    ELASTIC_L1_RATIO = 0.5
    
    # Ensemble / Bagging
    RF_N_ESTIMATORS = 100
    RF_MAX_DEPTH = 15
    RF_MIN_SAMPLES_SPLIT = 5
    
    # Boosting Models
    GB_N_ESTIMATORS = 100
    GB_LEARNING_RATE = 0.05
    GB_MAX_DEPTH = 5
    
    XGB_N_ESTIMATORS = 200
    XGB_LEARNING_RATE = 0.05
    XGB_MAX_DEPTH = 6
    
    LGB_N_ESTIMATORS = 200
    LGB_LEARNING_RATE = 0.05
    LGB_NUM_LEAVES = 31
    
    CAT_ITERATIONS = 200
    CAT_LEARNING_RATE = 0.05
    CAT_DEPTH = 6


# Initialize config
config = Config()

# Create directories
for dir_path in [config.MODEL_DIR, config.DATA_DIR]:
    os.makedirs(dir_path, exist_ok=True)

print("✅ Configuration initialized!")
print(f"📁 Model directory: {config.MODEL_DIR}")
print(f"📁 Data directory:  {config.DATA_DIR}")
print(f"🎲 Random state:   {config.RANDOM_STATE} (reproducible)")

✅ Configuration initialized!
📁 Model directory: models_iso
📁 Data directory:  data_iso
🎲 Random state:   42 (reproducible)


In [3]:
import os
import time
import requests
import pandas as pd

def download_with_retry(year, save_dir=config.DATA_DIR, max_retries=3, backoff_factor=2):
    """
    Attempts to download ISO-NE data with a retry mechanism.
    Gracefully falls back from .xlsx to .xls if a 404 is encountered.
    """
    extensions = ['.xlsx', '.xls']
    headers = {'User-Agent': 'Mozilla/5.0'} # Required to prevent server rejection
    
    for ext in extensions:
        # Construct the dynamic URL
        url = f"https://www.iso-ne.com/static-assets/documents/{year}/smd_hourly_{year}{ext}"
        file_path = os.path.join(save_dir, f"smd_hourly_{year}{ext}")
        
        # Skip if already downloaded
        if os.path.exists(file_path):
            print(f"✅ {year} already cached: {file_path}")
            return file_path

        print(f"🌐 Fetching {year} data from {url}...")
        
        # The Retry Mechanism
        for attempt in range(max_retries):
            try:
                response = requests.get(url, headers=headers, timeout=15)
                
                if response.status_code == 200:
                    with open(file_path, 'wb') as f:
                        f.write(response.content)
                    print(f"  ✅ Download successful.")
                    return file_path
                elif response.status_code == 404:
                    # File doesn't exist at this extension, break retry loop and try the next extension
                    print(f"  ⚠️ 404 Not Found for {ext}. Trying alternative extension...")
                    break 
                else:
                    print(f"  ⚠️ Server error {response.status_code}. Retrying in {backoff_factor * (attempt + 1)}s...")
                    
            except requests.exceptions.RequestException as e:
                print(f"  ⚠️ Network error: {e}. Retrying in {backoff_factor * (attempt + 1)}s...")
            
            # Exponential backoff sleep (e.g., 2s, 4s, 6s) before retrying
            time.sleep(backoff_factor * (attempt + 1))
            
    print(f"❌ Completely failed to download {year} after exhausting options.")
    return None

def build_master_parquet(years_list, data_dir):
    """
    Orchestrates the downloading of multiple years, determines the correct
    Pandas engine, concatenates them, and exports to a single Parquet file.
    """
    all_dfs = []
    print(f"🚀 Starting ingestion pipeline for years: {years_list}\n" + "-"*40)
    
    for year in years_list:
        # Step 1: Download
        file_path = download_with_retry(year, data_dir)
        
        if not file_path:
            continue # Skip to the next year if download totally failed
            
        # Step 2: Determine Engine & Load
        engine = 'openpyxl' if file_path.endswith('.xlsx') else 'xlrd'
        print(f"⏳ Reading {file_path} using {engine}...")
        
        try:
            df = pd.read_excel(file_path, sheet_name='ISO NE CA', engine=engine)
            all_dfs.append(df)
            print(f"  ✅ Added {df.shape[0]:,} rows to memory.\n")
        except Exception as e:
            print(f"  ❌ Error parsing {file_path}: {e}\n")
    
    # Step 3: Concatenate and Save
    if not all_dfs:
        print("❌ No data was successfully processed. Pipeline halted.")
        return None
        
    print("-" * 40 + "\n🔗 Concatenating DataFrames...")
    master_df = pd.concat(all_dfs, ignore_index=True)
    
    parquet_path = os.path.join(data_dir, "raw_iso_ne_master.parquet")
    master_df.to_parquet(parquet_path, index=False)
    
    print(f"✅ SUCCESS! Master Parquet created: {parquet_path}")
    print(f"📊 Total Pipeline Output: {master_df.shape[0]:,} rows, {master_df.shape[1]} columns")
    
    return parquet_path

# --- Execution ---
TARGET_YEARS = [2019, 2020, 2021, 2022, 2023] 

# Note: Uses config.DATA_DIR from your previous setup block
master_parquet = build_master_parquet(TARGET_YEARS, config.DATA_DIR)

if master_parquet:
    df_raw = pd.read_parquet(master_parquet)
    display(df_raw.head(3))

🚀 Starting ingestion pipeline for years: [2019, 2020, 2021, 2022, 2023]
----------------------------------------
🌐 Fetching 2019 data from https://www.iso-ne.com/static-assets/documents/2019/smd_hourly_2019.xlsx...
  ⚠️ 404 Not Found for .xlsx. Trying alternative extension...
🌐 Fetching 2019 data from https://www.iso-ne.com/static-assets/documents/2019/smd_hourly_2019.xls...
  ⚠️ 404 Not Found for .xls. Trying alternative extension...
❌ Completely failed to download 2019 after exhausting options.
🌐 Fetching 2020 data from https://www.iso-ne.com/static-assets/documents/2020/smd_hourly_2020.xlsx...
  ⚠️ 404 Not Found for .xlsx. Trying alternative extension...
🌐 Fetching 2020 data from https://www.iso-ne.com/static-assets/documents/2020/smd_hourly_2020.xls...
  ⚠️ 404 Not Found for .xls. Trying alternative extension...
❌ Completely failed to download 2020 after exhausting options.
🌐 Fetching 2021 data from https://www.iso-ne.com/static-assets/documents/2021/smd_hourly_2021.xlsx...
  ⚠️ 40